# Parte 0 — O piso

**Objetivo:** estabelecer um piso experimental treinando apenas com `rotulos_5.csv` (105 imagens) e comparar **(a) uma CNN pequena treinada do zero** com **(b) uma ResNet-18 pré-treinada no ImageNet com fine-tuning**. Os conjuntos de validação e `test_public` são usados para seleção de checkpoint e avaliação, respectivamente, não para treinamento.

In [125]:
from torch.utils.data import DataLoader
import pandas as pd
%load_ext autoreload
%autoreload 2
from sklearn.metrics import f1_score, accuracy_score, ConfusionMatrixDisplay
import torch
import torch.nn as nn
import numpy as np
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
# Importando a classe do seu arquivo dentro de src
from src.redimensionamento import DatasetTrap, train_epoch, transform_treino, transform_val, eval_model, CNN_pequena

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Reprodutibilidade

In [126]:
import random
torch.manual_seed(0); np.random.seed(0); random.seed(0)


## Dados

Leitura dos rótulos de treino, validação e teste público; codificação consistente das classes com `LabelEncoder` ajustado somente no treino.

In [127]:
df_treino = pd.read_csv("pa3_dados/labels/rotulos_5.csv")
df_val = pd.read_csv("pa3_dados/labels/val.csv")
df_teste = pd.read_csv('pa3_dados/labels/test_public.csv')
le = LabelEncoder()
df_treino['label'] = le.fit_transform(df_treino['label'])
df_val['label'] = le.transform(df_val['label'])
df_teste['label'] = le.transform(df_teste['label'])



In [128]:
dataset_treino = DatasetTrap(df=df_treino, raiz="pa3_dados/images/train", augment=transform_treino)
dataset_val = DatasetTrap(df=df_val, raiz="pa3_dados/images/val", augment=transform_val)
dataset_teste = DatasetTrap(df=df_teste, raiz = "pa3_dados/images/test_public", augment= transform_val  )

loader_val = DataLoader(dataset_val, batch_size=16, shuffle= False)
loader_train = DataLoader(dataset_treino, batch_size=16, shuffle= True)
loader_test = DataLoader(dataset_teste, batch_size= 16, shuffle= False)

In [129]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'xpu' if hasattr(torch, 'xpu') and torch.xpu.is_available() else 'cpu')

In [130]:
# Pesos de classe calculados somente a partir dos 105 exemplos de treino
rotulos_treino = df_treino['label'].values
pesos = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(rotulos_treino),
    y=rotulos_treino,
)
pesos_tensor = torch.tensor(pesos, dtype=torch.float32, device=DEVICE)

## (b) ResNet-18 pré-treinada com fine-tuning

In [131]:
modelo = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
num_entradas = modelo.fc.in_features
modelo.fc = nn.Linear(num_entradas, len(le.classes_))
modelo = modelo.to(DEVICE)
criterio = nn.CrossEntropyLoss(weight=pesos_tensor)

In [ ]:
history = train_epoch(model=modelo, device=DEVICE, criterio=criterio, train_loader=loader_train, val_loader=loader_val, label="resnet18_parte0")

Epoch 1/10 | F1: 0.1069 | Train Loss: 3.082286144438244 | Val Loss: 2.848291076055848 
Epoch 2/10 | F1: 0.2188 | Train Loss: 2.1312136695498514 | Val Loss: 2.57027805441677 


### Curva de treino e validação

A partir das curvas, observa-se indício de **sobreajuste por volta da época 7**, quando o modelo começa a decorar os exemplos de treino. A escolha do checkpoint é feita pela melhor F1 macro de validação, não pela última época.

In [ ]:
epocas = range(1, len(history['train_loss']) + 1)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 7), sharex=True)

# Loss
ax1.plot(epocas, history['train_loss'], label='Train')
ax1.plot(epocas, history['val_loss'], label='Validation')
ax1.set_ylabel('Loss')
ax1.set_title('Curva de treino e validação', fontweight='bold')
ax1.legend()

# F1 de validação
ax2.plot(epocas, history['f1_score'], color='tab:green', label='Val F1 macro')
melhor = int(np.argmax(history['f1_score']))
ax2.axvline(melhor + 1, color='gray', linestyle=':', label=f'Melhor época ({melhor + 1})')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('F1 macro')
ax2.legend()

for ax in (ax1, ax2):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

### Avaliação no test_public (checkpoint de melhor F1 no val)

In [ ]:
# Recupera o checkpoint escolhido pela melhor F1 macro na validação
pesos_salvos = torch.load('models/best_model_resnet18_parte0.pt', map_location=DEVICE, weights_only=True)
modelo.load_state_dict(pesos_salvos)
previsoes_teste, rotulos_teste = eval_model(modelo, loader_test, criterio, DEVICE)
print('F1 macro (b):', f1_score(rotulos_teste, previsoes_teste, average='macro'))
print('Acurácia (b):', accuracy_score(rotulos_teste, previsoes_teste))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))
ConfusionMatrixDisplay.from_predictions(
    rotulos_teste, previsoes_teste,
    labels=range(len(le.classes_)),
    display_labels=le.classes_,
    normalize='true',
    values_format='.2f',
    xticks_rotation=90,
    colorbar=False,
    cmap='Blues',
    ax=ax,
)
for t in ax.texts:
    t.set_fontsize(7)
ax.set_title('Matriz de confusão (normalizada por linha) — ResNet-18', fontweight='bold')
plt.tight_layout()
plt.show()

## (a) CNN pequena do zero

In [ ]:
modelo_CNN = CNN_pequena().to(DEVICE)
criterio_cnn = nn.CrossEntropyLoss(weight=pesos_tensor)

history_opcao_a = train_epoch(
    model=modelo_CNN,
    device=DEVICE,
    criterio=criterio_cnn,
    train_loader=loader_train,
    val_loader=loader_val,
    label="cnn_parte0",
    epochs=60,      # suba para 100 se a val ainda estiver melhorando
    lr=1e-3,
)
modelo_CNN.load_state_dict(torch.load('models/best_model_cnn_parte0.pt', map_location=DEVICE, weights_only=True))


previsoes_a, rotulos_a = eval_model(modelo_CNN, loader_test, criterio_cnn, DEVICE)
print("F1 macro:", f1_score(rotulos_a, previsoes_a, average='macro'))
print("Acurácia:", accuracy_score(rotulos_a, previsoes_a))

## Resumo do piso

Comparação dos dois modelos no mesmo conjunto `test_public`, usando F1 macro e acurácia.

In [ ]:
resumo = pd.DataFrame({
    'modelo': ['CNN do zero (a)', 'ResNet-18 ImageNet (b)'],
    'f1_macro': [f1_score(rotulos_a, previsoes_a, average='macro'),
                 f1_score(rotulos_teste, previsoes_teste, average='macro')],
    'acuracia': [accuracy_score(rotulos_a, previsoes_a),
                 accuracy_score(rotulos_teste, previsoes_teste)],
})
resumo